In [1]:
from nba_api.stats.endpoints import leaguegamelog, gamerotation, playbyplayv3
import pandas as pd
import numpy as np
from tqdm import tqdm
import time

In [2]:
# from nba_api.stats.static import teams

# nba_teams = teams.get_teams()
# # Select the dictionary for the Celtics, which contains their team ID
# celtics = [team for team in nba_teams if team['abbreviation'] == 'BOS'][0]
# celtics_id = celtics['id']

# Get Games in a Season

In [4]:
# from nba_api.stats.endpoints import leaguegamefinder

# # Query for games where the Celtics were playing
# gamefinder = leaguegamefinder.LeagueGameFinder(team_id_nullable=celtics_id)
# # The first DataFrame of those returned is what we want.
# games = gamefinder.get_data_frames()[0]
# games.head()

In [5]:
games_df = leaguegamelog.LeagueGameLog(player_or_team_abbreviation = 'T', season = '2023-24', season_type_all_star = 'Regular Season').get_data_frames()[0]
games_df.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22023,1610612743,DEN,Denver Nuggets,0022300061,2023-10-24,DEN vs. LAL,W,240,48,...,33,42,29,9,6,12,15,119,12,1
1,22023,1610612744,GSW,Golden State Warriors,0022300062,2023-10-24,GSW vs. PHX,L,240,36,...,31,49,19,11,6,11,23,104,-4,1
2,22023,1610612747,LAL,Los Angeles Lakers,0022300061,2023-10-24,LAL @ DEN,L,240,41,...,31,44,23,5,4,12,18,107,-12,1
3,22023,1610612756,PHX,Phoenix Suns,0022300062,2023-10-24,PHX @ GSW,W,240,42,...,43,60,23,5,7,19,22,108,4,1
4,22023,1610612737,ATL,Atlanta Hawks,0022300063,2023-10-25,ATL @ CHA,L,240,39,...,30,42,24,12,1,12,19,110,-6,1


# Get Rotation for a Given Game

In [7]:
ht_rotation = gamerotation.GameRotation(game_id = games_df.GAME_ID[0]).get_data_frames()[1]
ht_rotation['home_away_flg'] = 'H'
at_rotation = gamerotation.GameRotation(game_id = games_df.GAME_ID[0]).get_data_frames()[0]
at_rotation['home_away_flg'] = 'A'
rotation_df = pd.concat([ht_rotation, at_rotation])
rotation_df.head()

,GAME_ID,TEAM_ID,TEAM_CITY,TEAM_NAME,PERSON_ID,PLAYER_FIRST,PLAYER_LAST,IN_TIME_REAL,OUT_TIME_REAL,PLAYER_PTS,PT_DIFF,USG_PCT,home_away_flg
0,0022300061,1610612743,Denver,Nuggets,202704,Reggie,Jackson,3530.0,8890.0,8,14.0,0.278,H
1,0022300061,1610612743,Denver,Nuggets,202704,Reggie,Jackson,9220.0,11940.0,0,-4.0,0.273,H
2,0022300061,1610612743,Denver,Nuggets,202704,Reggie,Jackson,18510.0,24870.0,0,1.0,0.095,H
3,0022300061,1610612743,Denver,Nuggets,203484,Kentavious,Caldwell-Pope,0.0,5500.0,5,11.0,0.174,H
4,0022300061,1610612743,Denver,Nuggets,203484,Kentavious,Caldwell-Pope,8890.0,20620.0,11,-6.0,0.204,H


In [8]:
rotation_df = rotation_df.sort_values(by = ['IN_TIME_REAL', 'OUT_TIME_REAL'], ascending = [True, True])
rotation_df = rotation_df.reset_index(drop = True)
rotation_df['IN_TIME_REAL'] = np.ceil(rotation_df['IN_TIME_REAL']/10) #Divide by 10 to get actual seconds
rotation_df['OUT_TIME_REAL'] = np.ceil(rotation_df['OUT_TIME_REAL']/10) #Divide by 10 to get actual seconds
rotation_df.head()

,GAME_ID,TEAM_ID,TEAM_CITY,TEAM_NAME,PERSON_ID,PLAYER_FIRST,PLAYER_LAST,IN_TIME_REAL,OUT_TIME_REAL,PLAYER_PTS,PT_DIFF,USG_PCT,home_away_flg
0,0022300061,1610612743,Denver,Nuggets,1627750,Jamal,Murray,0.0,353.0,5,4.0,0.267,H
1,0022300061,1610612747,L.A. Lakers,Lakers,2544,LeBron,James,0.0,353.0,4,-4.0,0.200,A
2,0022300061,1610612747,L.A. Lakers,Lakers,1626156,D'Angelo,Russell,0.0,353.0,0,-4.0,0.200,A
3,0022300061,1610612743,Denver,Nuggets,203484,Kentavious,Caldwell-Pope,0.0,550.0,5,11.0,0.174,H
4,0022300061,1610612743,Denver,Nuggets,203932,Aaron,Gordon,0.0,550.0,2,11.0,0.043,H


# Get PBP Stats for a Given Game

In [10]:
pbp = playbyplayv3.PlayByPlayV3(game_id = games_df.GAME_ID[0]).get_data_frames()[0]
pbp.head()

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,...,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0022300061,2,PT12M00.00S,1,0,,0,,,0,...,0,0,0,,Start of 1st Period (7:36 PM EST),period,start,0,0,1
1,0022300061,4,PT12M00.00S,1,1610612743,DEN,203999,Jokić,N. Jokić,0,...,,,0,h,Jump Ball Jokic vs. Davis: Tip to James,Jump Ball,,1,0,2
2,0022300061,7,PT11M42.00S,1,1610612747,LAL,203076,Davis,A. Davis,2,...,0,2,2,v,Davis 1' Dunk (2 PTS) (Russell 1 AST),Made Shot,Dunk Shot,1,2,3
3,0022300061,10,PT11M15.00S,1,1610612743,DEN,203999,Jokić,N. Jokić,13,...,2,2,4,h,Jokic 7' Driving Floating Jump Shot (2 PTS) (M...,Made Shot,Driving Floating Jump Shot,1,2,4
4,0022300061,11,PT10M57.00S,1,1610612747,LAL,1627752,Prince,T. Prince,237,...,2,5,7,v,Prince 24' 3PT Jump Shot (3 PTS) (James 1 AST),Made Shot,Jump Shot,1,3,5


## Calculate Game Breakpoints

In [12]:
breakpoints = sorted(set(rotation_df['IN_TIME_REAL']).union(set(rotation_df['OUT_TIME_REAL'])))
breakpoints

[0.0,
 353.0,
 550.0,
 680.0,
 720.0,
 889.0,
 898.0,
 922.0,
 1015.0,
 1045.0,
 1194.0,
 1313.0,
 1440.0,
 1756.0,
 1851.0,
 1868.0,
 1925.0,
 2062.0,
 2160.0,
 2351.0,
 2352.0,
 2385.0,
 2474.0,
 2487.0,
 2516.0,
 2805.0,
 2837.0,
 2880.0]

In [13]:
len(breakpoints)

28

## Calculate Stints and Point Differential

In [15]:
#Get Actual Point Differential
pbp['time_completed'] = ((pbp['period']-1) * 12*60) + ((12*60) - pbp['clock'].str[2:4].astype(float)*60 - pbp['clock'].str[5:7].astype(float))
pbp_formatted = pbp[['clock', 'time_completed', 'scoreHome', 'scoreAway', 'description', 'actionType']].copy()
pbp_formatted['scoreHome'].replace('', np.nan, inplace = True)
pbp_formatted['scoreAway'].replace('', np.nan, inplace = True)
pbp_formatted['scoreHome'].fillna(method = 'ffill', inplace = True)
pbp_formatted['scoreAway'].fillna(method = 'ffill', inplace = True)

pbp_formatted['scoreHome'] = pbp_formatted['scoreHome'].astype(int)
pbp_formatted['scoreAway'] = pbp_formatted['scoreAway'].astype(int)




pbp_formatted['STINT_START_TIME'] = pbp_formatted['time_completed'].shift(1)
pbp_formatted['STINT_START_TIME'] = pbp_formatted['STINT_START_TIME'].fillna(0)

pbp_formatted.rename(columns = {'time_completed': 'STINT_END_TIME'}, inplace = True)
pbp_formatted = pbp_formatted.loc[pbp_formatted['STINT_END_TIME'].isin(breakpoints)]


pt_diff_df = pd.DataFrame(columns = ['STINT_START_TIME', 'STINT_END_TIME', 'PT_DIFF'])

for i in range(len(breakpoints) - 1):
    t = breakpoints[i]
    t_plus_1 = breakpoints[i+1]

    temp = pbp_formatted.loc[(pbp_formatted['STINT_START_TIME'] >= t) & (pbp_formatted['STINT_END_TIME'] <= t_plus_1)]
    res = pd.DataFrame(columns = ['STINT_START_TIME', 'STINT_END_TIME', 'PT_DIFF'])

    res.loc[0, 'STINT_START_TIME'] = t 
    res.loc[0, 'STINT_END_TIME'] = t_plus_1

    diff = 0

    if t == 0:
        diff = np.max(temp['scoreHome']) - np.max(temp['scoreAway'])
    
    else:
        end_home_score = temp.loc[(temp['STINT_END_TIME'] == t_plus_1) & (temp['actionType'].isin(['Substitution', 'period']))]['scoreHome'].values[0]
        end_away_score = temp.loc[(temp['STINT_END_TIME'] == t_plus_1) & (temp['actionType'].isin(['Substitution', 'period']))]['scoreAway'].values[0]
        # diff = (np.max(temp['scoreHome']) - np.min(temp['scoreHome'])) - (np.max(temp['scoreAway']) - np.min(temp['scoreAway']))
        diff = (end_home_score - np.min(temp['scoreHome'])) - (end_away_score - np.min(temp['scoreAway']))
    
    res.loc[0, 'PT_DIFF'] = diff 

    pt_diff_df = pd.concat([pt_diff_df, res], axis = 0)
    pt_diff_df = pt_diff_df.reset_index(drop = True)

pt_diff_df

/var/folders/bl/md9bs_8x7v31z42n_0_459yw0000gn/T/ipykernel_44690/139070337.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  pbp_formatted['scoreHome'].replace('', np.nan, inplace = True)
/var/folders/bl/md9bs_8x7v31z42n_0_459yw0000gn/T/ipykernel_44690/139070337.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting

,STINT_START_TIME,STINT_END_TIME,PT_DIFF
0,0.0,353.0,4
1,353.0,550.0,7
2,550.0,680.0,-1
3,680.0,720.0,4
4,720.0,889.0,4
5,889.0,898.0,0
6,898.0,922.0,-2
7,922.0,1015.0,-4
8,1015.0,1045.0,1
9,1045.0,1194.0,0


# Get In-Time and Out-Time for each Player

In [17]:
players = list(set(rotation_df.PERSON_ID))
# players = ['PERSON_ID_'+str(i) for i in players]
players

[1629060,
 1631108,
 1631128,
 203932,
 1629216,
 1626156,
 1629618,
 1629629,
 1626174,
 203076,
 1629637,
 1629008,
 202704,
 1630296,
 203484,
 1630559,
 203999,
 1627750,
 1627752,
 1631212,
 2544,
 1630192,
 1631221,
 1641721]

In [18]:
player_id_cols = ['PERSON_ID_'+str(p) for p in players]
player_flags_df = pd.get_dummies(rotation_df, columns = ['PERSON_ID'], dtype = int).copy()
player_flags_df = player_flags_df.groupby(['IN_TIME_REAL', 'OUT_TIME_REAL'])[player_id_cols].sum().reset_index()
player_flags_df

,IN_TIME_REAL,OUT_TIME_REAL,PERSON_ID_1629060,PERSON_ID_1631108,PERSON_ID_1631128,PERSON_ID_203932,PERSON_ID_1629216,PERSON_ID_1626156,PERSON_ID_1629618,PERSON_ID_1629629,...,PERSON_ID_203484,PERSON_ID_1630559,PERSON_ID_203999,PERSON_ID_1627750,PERSON_ID_1627752,PERSON_ID_1631212,PERSON_ID_2544,PERSON_ID_1630192,PERSON_ID_1631221,PERSON_ID_1641721
0,0.0,353.0,0,0,0,0,0,1,0,0,...,0,0,0,1,0,0,1,0,0,0
1,0.0,550.0,0,0,0,1,0,0,0,0,...,1,1,1,0,1,0,0,0,0,0
2,353.0,680.0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,353.0,889.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,353.0,898.0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,550.0,720.0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
6,550.0,889.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,550.0,922.0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,1,0,0
8,550.0,1015.0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
9,550.0,1045.0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Final Stint-wise Point Differential DataFrame

In [20]:
final_stints_df = pd.DataFrame(columns = ['STINT_START_TIME', 'STINT_END_TIME'] + players + ['PT_DIFF'])

for i in tqdm(range(len(breakpoints) - 1)):
  t = breakpoints[i]
  t_plus_1 = breakpoints[i+1]
  stint_players = rotation_df.loc[(rotation_df['IN_TIME_REAL'] <= t) & (rotation_df['OUT_TIME_REAL'] > t)][['PERSON_ID']].values.flatten().tolist()
  home_away_flgs = rotation_df.loc[(rotation_df['IN_TIME_REAL'] <= t) & (rotation_df['OUT_TIME_REAL'] > t)][['home_away_flg']].values.tolist()
  # pt_diff = stints_df.loc[(stints_df['STINT_START_TIME_FIXED'] == t) & (stints_df['STINT_END_TIME'] == t_plus_1)]['PT_DIFF'].sum()
  pt_diff = pt_diff_df.loc[(pt_diff_df['STINT_START_TIME'] == t) & (pt_diff_df['STINT_END_TIME'] == t_plus_1)]['PT_DIFF'].values[0]

  X = pd.DataFrame(columns = ['STINT_START_TIME', 'STINT_END_TIME'] + players + ['PT_DIFF'])
  X.loc[0, 'STINT_START_TIME'] = t
  X.loc[0, 'STINT_END_TIME'] = t_plus_1
  X.loc[0, 'PT_DIFF'] = pt_diff

  for player in players:
    if player in stint_players:
      if home_away_flgs[stint_players.index(player)] == 'H':
        X.loc[0, player] = 1
      else:
        X.loc[0, player] = -1
    else:
      X.loc[0, player] = 0

  final_stints_df = pd.concat([final_stints_df, X], axis = 0)

final_stints_df = final_stints_df.reset_index(drop = True)
final_stints_df

100%|██████████████████████████████████████████| 27/27 [00:00<00:00, 674.00it/s]


,STINT_START_TIME,STINT_END_TIME,1629060,1631108,1631128,203932,1629216,1626156,1629618,1629629,...,1630559,203999,1627750,1627752,1631212,2544,1630192,1631221,1641721,PT_DIFF
0,0.0,353.0,0,0,0,-1,0,-1,0,0,...,-1,-1,-1,-1,0,-1,0,0,0,4
1,353.0,550.0,-1,0,0,-1,-1,0,0,0,...,-1,-1,0,-1,0,0,0,0,0,7
2,550.0,680.0,-1,0,-1,0,-1,-1,0,-1,...,0,0,-1,0,-1,0,-1,0,0,-1
3,680.0,720.0,0,0,-1,0,-1,-1,0,-1,...,0,0,-1,0,-1,-1,-1,0,0,4
4,720.0,889.0,0,0,-1,0,-1,-1,0,0,...,0,0,-1,0,-1,-1,-1,0,0,4
5,889.0,898.0,0,0,-1,0,-1,-1,0,0,...,0,0,-1,0,-1,-1,-1,0,0,0
6,898.0,922.0,0,0,-1,0,0,-1,0,-1,...,0,0,-1,0,-1,-1,-1,0,0,-2
7,922.0,1015.0,0,0,-1,-1,0,-1,0,-1,...,0,-1,0,0,0,-1,0,0,0,-4
8,1015.0,1045.0,0,0,-1,-1,0,0,0,-1,...,-1,-1,0,0,0,-1,0,0,0,1
9,1045.0,1194.0,0,0,0,-1,0,0,0,-1,...,-1,-1,0,-1,0,0,0,0,0,0


In [21]:
np.sum(final_stints_df['PT_DIFF'])

12

# Scraping Definition

## GameLog to GameID

In [24]:
def gamelog_gameid(szn:str):
    X = leaguegamelog.LeagueGameLog(player_or_team_abbreviation = 'T', season = szn, season_type_all_star = 'Regular Season').get_data_frames()[0]
    return X, set(X['GAME_ID'])

In [25]:
gamelog_gameid('2024-25')

(     SEASON_ID     TEAM_ID TEAM_ABBREVIATION               TEAM_NAME  \
 0        22024  1610612738               BOS          Boston Celtics   
 1        22024  1610612747               LAL      Los Angeles Lakers   
 2        22024  1610612750               MIN  Minnesota Timberwolves   
 3        22024  1610612752               NYK         New York Knicks   
 4        22024  1610612737               ATL           Atlanta Hawks   
 ...        ...         ...               ...                     ...   
 2455     22024  1610612762               UTA               Utah Jazz   
 2456     22024  1610612763               MEM       Memphis Grizzlies   
 2457     22024  1610612764               WAS      Washington Wizards   
 2458     22024  1610612765               DET         Detroit Pistons   
 2459     22024  1610612766               CHA       Charlotte Hornets   
 
          GAME_ID   GAME_DATE      MATCHUP WL  MIN  FGM  ...  DREB  REB  AST  \
 0     0022400061  2024-10-22  BOS vs. NYK

## Rotation Based on GameID

In [27]:
def get_rotation(game_id:str):
    full_rotation = gamerotation.GameRotation(game_id = game_id).get_data_frames()
    ht_rotation = full_rotation[1]
    ht_rotation['home_away_flg'] = 'H'
    at_rotation = full_rotation[0]
    at_rotation['home_away_flg'] = 'A'
    rotation_df = pd.concat([ht_rotation, at_rotation])
    rotation_df = rotation_df.sort_values(by = ['IN_TIME_REAL', 'OUT_TIME_REAL'], ascending = [True, True])
    rotation_df = rotation_df.reset_index(drop = True)
    rotation_df['IN_TIME_REAL'] = np.ceil(rotation_df['IN_TIME_REAL']/10)
    rotation_df['OUT_TIME_REAL'] = np.ceil(rotation_df['OUT_TIME_REAL']/10)

    #Calc. Breakpoints
    breakpoints = sorted(set(rotation_df['IN_TIME_REAL']).union(set(rotation_df['OUT_TIME_REAL'])))

    #Add Sleep Time
    time.sleep(1)

    return rotation_df, breakpoints

In [28]:
rotation_df, breakpoints = get_rotation('0022400061')

In [29]:
rotation_df

,GAME_ID,TEAM_ID,TEAM_CITY,TEAM_NAME,PERSON_ID,PLAYER_FIRST,PLAYER_LAST,IN_TIME_REAL,OUT_TIME_REAL,PLAYER_PTS,PT_DIFF,USG_PCT,home_away_flg
0,0022400061,1610612738,Boston,Celtics,201950,Jrue,Holiday,0.0,325.0,0,7.0,0.067,H
1,0022400061,1610612738,Boston,Celtics,1627759,Jaylen,Brown,0.0,325.0,5,7.0,0.333,H
2,0022400061,1610612752,New York,Knicks,1626157,Karl-Anthony,Towns,0.0,325.0,2,-7.0,0.182,A
3,0022400061,1610612752,New York,Knicks,1628384,OG,Anunoby,0.0,416.0,2,-7.0,0.133,A
4,0022400061,1610612738,Boston,Celtics,201143,Al,Horford,0.0,429.0,3,5.0,0.050,H
5,0022400061,1610612738,Boston,Celtics,1628401,Derrick,White,0.0,559.0,5,13.0,0.217,H
6,0022400061,1610612752,New York,Knicks,1628404,Josh,Hart,0.0,559.0,5,-13.0,0.300,A
7,0022400061,1610612752,New York,Knicks,1628973,Jalen,Brunson,0.0,559.0,6,-13.0,0.250,A
8,0022400061,1610612738,Boston,Celtics,1628369,Jayson,Tatum,0.0,720.0,15,19.0,0.300,H
9,0022400061,1610612752,New York,Knicks,1628969,Mikal,Bridges,0.0,807.0,0,-15.0,0.179,A


In [30]:
breakpoints

[0.0,
 325.0,
 416.0,
 429.0,
 559.0,
 720.0,
 807.0,
 1062.0,
 1095.0,
 1245.0,
 1834.0,
 1973.0,
 2101.0,
 2160.0,
 2286.0,
 2365.0,
 2515.0,
 2556.0,
 2620.0,
 2713.0,
 2880.0]

## Stints and Point Differential from PBP Data

In [32]:
def get_stints_pt_diff(game_id:str, breakpoints:list):
    pbp = playbyplayv3.PlayByPlayV3(game_id = game_id).get_data_frames()[0]

    #Get Actual Point Differential
    pbp['time_completed'] = ((pbp['period']-1) * 12*60) + ((12*60) - pbp['clock'].str[2:4].astype(float)*60 - pbp['clock'].str[5:7].astype(float))
    pbp_formatted = pbp[['clock', 'time_completed', 'scoreHome', 'scoreAway', 'description', 'actionType']].copy()
    pbp_formatted['scoreHome'].replace('', np.nan, inplace = True)
    pbp_formatted['scoreAway'].replace('', np.nan, inplace = True)
    pbp_formatted['scoreHome'].fillna(method = 'ffill', inplace = True)
    pbp_formatted['scoreAway'].fillna(method = 'ffill', inplace = True)
    
    pbp_formatted['scoreHome'] = pbp_formatted['scoreHome'].astype(int)
    pbp_formatted['scoreAway'] = pbp_formatted['scoreAway'].astype(int)
    
    
    
    
    pbp_formatted['STINT_START_TIME'] = pbp_formatted['time_completed'].shift(1)
    pbp_formatted['STINT_START_TIME'] = pbp_formatted['STINT_START_TIME'].fillna(0)
    
    pbp_formatted.rename(columns = {'time_completed': 'STINT_END_TIME'}, inplace = True)
    pbp_formatted = pbp_formatted.loc[pbp_formatted['STINT_END_TIME'].isin(breakpoints)]
    
    
    pt_diff_df = pd.DataFrame(columns = ['STINT_START_TIME', 'STINT_END_TIME', 'PT_DIFF'])
    
    for i in range(len(breakpoints) - 1):
        t = breakpoints[i]
        t_plus_1 = breakpoints[i+1]
    
        temp = pbp_formatted.loc[(pbp_formatted['STINT_START_TIME'] >= t) & (pbp_formatted['STINT_END_TIME'] <= t_plus_1)]
        res = pd.DataFrame(columns = ['STINT_START_TIME', 'STINT_END_TIME', 'PT_DIFF'])
    
        res.loc[0, 'STINT_START_TIME'] = t 
        res.loc[0, 'STINT_END_TIME'] = t_plus_1
    
        diff = 0
    
        if t == 0:
            diff = np.max(temp['scoreHome']) - np.max(temp['scoreAway'])
        
        else:
            end_home_score = temp.loc[(temp['STINT_END_TIME'] == t_plus_1) & (temp['actionType'].isin(['Substitution', 'period']))]['scoreHome'].values[0]
            end_away_score = temp.loc[(temp['STINT_END_TIME'] == t_plus_1) & (temp['actionType'].isin(['Substitution', 'period']))]['scoreAway'].values[0]
            # diff = (np.max(temp['scoreHome']) - np.min(temp['scoreHome'])) - (np.max(temp['scoreAway']) - np.min(temp['scoreAway']))
            diff = (end_home_score - np.min(temp['scoreHome'])) - (end_away_score - np.min(temp['scoreAway']))
        
        res.loc[0, 'PT_DIFF'] = diff 
    
        pt_diff_df = pd.concat([pt_diff_df, res], axis = 0)
        pt_diff_df = pt_diff_df.reset_index(drop = True)

    #Add Sleep Time
    time.sleep(1)
    
    return pt_diff_df

In [33]:
pt_diff_df = get_stints_pt_diff('0022400061', breakpoints)
pt_diff_df

/var/folders/bl/md9bs_8x7v31z42n_0_459yw0000gn/T/ipykernel_44690/3001117299.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  pbp_formatted['scoreHome'].replace('', np.nan, inplace = True)
/var/folders/bl/md9bs_8x7v31z42n_0_459yw0000gn/T/ipykernel_44690/3001117299.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setti

,STINT_START_TIME,STINT_END_TIME,PT_DIFF
0,0.0,325.0,7
1,325.0,416.0,1
2,416.0,429.0,-2
3,429.0,559.0,8
4,559.0,720.0,6
5,720.0,807.0,-4
6,807.0,1062.0,4
7,1062.0,1095.0,-1
8,1095.0,1245.0,5
9,1245.0,1834.0,6


## PlayerID In-Time Out-Time

In [35]:
def get_player_id_in_out_time(rotation_df:pd.DataFrame):
    players = list(set(rotation_df.PERSON_ID))

    player_id_cols = ['PERSON_ID_'+str(p) for p in players]
    player_flags_df = pd.get_dummies(rotation_df, columns = ['PERSON_ID'], dtype = int).copy()
    player_flags_df = player_flags_df.groupby(['IN_TIME_REAL', 'OUT_TIME_REAL'])[player_id_cols].sum().reset_index()
    return players, player_flags_df

In [36]:
players, _ = get_player_id_in_out_time(rotation_df)
players

[1630214,
 1628436,
 1642278,
 1628969,
 1626157,
 1628973,
 1641775,
 1626166,
 201143,
 1630540,
 1628369,
 201950,
 1628384,
 1629674,
 1631210,
 1630573,
 1630574,
 1627759,
 1628401,
 1630579,
 1628404,
 1642359,
 1630202]

## Final Stint-wise Point Differential Function

In [38]:
def final_stint_pt_diff(players:list, breakpoints:list, rotation_df:pd.DataFrame, pt_diff_df:pd.DataFrame, game_id:str, season:str):
    final_stints_df = pd.DataFrame(columns = ['SEASON', 'GAME_ID', 'STINT_START_TIME', 'STINT_END_TIME'] + players + ['PT_DIFF'])

    for i in tqdm(range(len(breakpoints) - 1)):
      t = breakpoints[i]
      t_plus_1 = breakpoints[i+1]
      stint_players = rotation_df.loc[(rotation_df['IN_TIME_REAL'] <= t) & (rotation_df['OUT_TIME_REAL'] > t)][['PERSON_ID']].values.flatten().tolist()
      home_away_flgs = rotation_df.loc[(rotation_df['IN_TIME_REAL'] <= t) & (rotation_df['OUT_TIME_REAL'] > t)][['home_away_flg']].values.tolist()
      # pt_diff = stints_df.loc[(stints_df['STINT_START_TIME_FIXED'] == t) & (stints_df['STINT_END_TIME'] == t_plus_1)]['PT_DIFF'].sum()
      pt_diff = pt_diff_df.loc[(pt_diff_df['STINT_START_TIME'] == t) & (pt_diff_df['STINT_END_TIME'] == t_plus_1)]['PT_DIFF'].values[0]
    
      X = pd.DataFrame(columns = ['SEASON', 'GAME_ID', 'STINT_START_TIME', 'STINT_END_TIME'] + players + ['PT_DIFF'])
      #Assign Season and GameID
      X.loc[0, 'SEASON'] = season
      X.loc[0, 'GAME_ID'] = game_id
      X.loc[0, 'STINT_START_TIME'] = t
      X.loc[0, 'STINT_END_TIME'] = t_plus_1
      X.loc[0, 'PT_DIFF'] = pt_diff
    
      for player in players:
        if player in stint_players:
          if home_away_flgs[stint_players.index(player)] == 'H':
            X.loc[0, player] = 1
          else:
            X.loc[0, player] = -1
        else:
          X.loc[0, player] = 0
    
      final_stints_df = pd.concat([final_stints_df, X], axis = 0)
    
    final_stints_df = final_stints_df.reset_index(drop = True)
    return final_stints_df

In [39]:
final_stint_pt_diff(players, breakpoints, rotation_df, pt_diff_df, '0022400061', '2024-25')

100%|██████████████████████████████████████████| 20/20 [00:00<00:00, 766.64it/s]


,SEASON,GAME_ID,STINT_START_TIME,STINT_END_TIME,1630214,1628436,1642278,1628969,1626157,1628973,...,1631210,1630573,1630574,1627759,1628401,1630579,1628404,1642359,1630202,PT_DIFF
0,2024-25,0022400061,0.0,325.0,0,0,0,-1,-1,-1,...,0,0,0,-1,-1,0,-1,0,0,7
1,2024-25,0022400061,325.0,416.0,0,-1,0,-1,0,-1,...,0,0,0,0,-1,-1,-1,0,-1,1
2,2024-25,0022400061,416.0,429.0,0,-1,0,-1,0,-1,...,0,0,0,0,-1,-1,-1,0,-1,-2
3,2024-25,0022400061,429.0,559.0,0,-1,0,-1,0,-1,...,0,-1,0,0,-1,-1,-1,0,-1,8
4,2024-25,0022400061,559.0,720.0,0,-1,0,-1,-1,0,...,0,-1,0,0,0,0,0,0,-1,6
5,2024-25,0022400061,720.0,807.0,-1,0,0,-1,-1,0,...,0,-1,0,-1,0,0,0,0,0,-4
6,2024-25,0022400061,807.0,1062.0,-1,0,0,0,-1,0,...,0,-1,0,-1,0,0,0,-1,0,4
7,2024-25,0022400061,1062.0,1095.0,0,-1,0,0,0,-1,...,0,0,0,-1,-1,-1,-1,-1,-1,-1
8,2024-25,0022400061,1095.0,1245.0,0,-1,0,-1,0,-1,...,0,0,0,-1,-1,-1,-1,0,-1,5
9,2024-25,0022400061,1245.0,1834.0,0,0,0,-1,-1,-1,...,0,0,0,-1,-1,0,-1,0,0,6


## Process Game

In [41]:
def process_game(game_id: str, szn: int) -> pd.DataFrame:
    """Compute one game's stint-wise point differential dataframe. Also provides the point differential per 48 minutes."""
    # If you still need to throttle a bit to be polite to an API, uncomment:
    # time.sleep(0.10)

    rotation_df, breakpoints = get_rotation(game_id)                             # Get Rotation and Breakpoints
    pt_diff_df = get_stints_pt_diff(game_id, breakpoints)                        # Get Point Differential for each Stint
    players, _ = get_player_id_in_out_time(rotation_df)                          # Get List of Players
    game_res_df = final_stint_pt_diff(players, breakpoints, rotation_df,
                                      pt_diff_df, game_id, szn)                  # Final per-game DF

    
    game_res_df['STINT_DURATION'] = game_res_df['STINT_END_TIME'] - game_res_df['STINT_START_TIME']                                          # Stint Duration in seconds
    game_res_df['PT_DIFF_PER_48'] = (48 * 60 * game_res_df['PT_DIFF'])/(game_res_df['STINT_END_TIME'] - game_res_df['STINT_START_TIME'])     # Final per48 Point Differential
    return game_res_df

In [71]:
process_game('0022400061', '2024-25')

/var/folders/bl/md9bs_8x7v31z42n_0_459yw0000gn/T/ipykernel_44690/3001117299.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  pbp_formatted['scoreHome'].replace('', np.nan, inplace = True)
/var/folders/bl/md9bs_8x7v31z42n_0_459yw0000gn/T/ipykernel_44690/3001117299.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setti

,SEASON,GAME_ID,STINT_START_TIME,STINT_END_TIME,1630214,1628436,1642278,1628969,1626157,1628973,...,1630574,1627759,1628401,1630579,1628404,1642359,1630202,PT_DIFF,STINT_DURATION,PT_DIFF_PER_48
0,2024-25,0022400061,0.0,325.0,0,0,0,-1,-1,-1,...,0,-1,-1,0,-1,0,0,7,325.0,62.030769
1,2024-25,0022400061,325.0,416.0,0,-1,0,-1,0,-1,...,0,0,-1,-1,-1,0,-1,1,91.0,31.648352
2,2024-25,0022400061,416.0,429.0,0,-1,0,-1,0,-1,...,0,0,-1,-1,-1,0,-1,-2,13.0,-443.076923
3,2024-25,0022400061,429.0,559.0,0,-1,0,-1,0,-1,...,0,0,-1,-1,-1,0,-1,8,130.0,177.230769
4,2024-25,0022400061,559.0,720.0,0,-1,0,-1,-1,0,...,0,0,0,0,0,0,-1,6,161.0,107.329193
5,2024-25,0022400061,720.0,807.0,-1,0,0,-1,-1,0,...,0,-1,0,0,0,0,0,-4,87.0,-132.413793
6,2024-25,0022400061,807.0,1062.0,-1,0,0,0,-1,0,...,0,-1,0,0,0,-1,0,4,255.0,45.176471
7,2024-25,0022400061,1062.0,1095.0,0,-1,0,0,0,-1,...,0,-1,-1,-1,-1,-1,-1,-1,33.0,-87.272727
8,2024-25,0022400061,1095.0,1245.0,0,-1,0,-1,0,-1,...,0,-1,-1,-1,-1,0,-1,5,150.0,96.0
9,2024-25,0022400061,1245.0,1834.0,0,0,0,-1,-1,-1,...,0,-1,-1,0,-1,0,0,6,589.0,29.337861


## Pulling All Functions together to Scrape for Multiple Games

In [44]:
final_df = pd.DataFrame(columns = ['SEASON', 'GAME_ID', 'STINT_START_TIME', 'STINT_END_TIME', 'PT_DIFF'])

In [45]:
SEASONS = ['2022-23', '2023-24', '2024-25']

### Serial Processing

In [47]:
# for szn in SEASONS:
#     games, game_ids = gamelog_gameid(szn)

#     print(f"Now Running Season: {szn}")

#     for game_id in tqdm(game_ids):
#         rotation_df, breakpoints = get_rotation(game_id) #Get Rotation and Breakpoints
#         pt_diff_df = get_stints_pt_diff(game_id, breakpoints) #Get Point Differential for each Stint
#         players, _ = get_player_id_in_out_time(rotation_df) #Get List of Players

#         game_res_df = final_stint_pt_diff(players, breakpoints, rotation_df, pt_diff_df, game_id, szn) #Final Stint-Wise Point Differential for the Game with Player Flags

#         final_df = pd.concat([final_df, game_res_df], axis = 0, ignore_index=True)


#         time.sleep(1) #Sleep 1 second in between games

#     time.sleep(5) #Sleep 5 Seconds in-between seasons

### Parallel Processing

In [49]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings("ignore")

# Tune this based on your API limits / machine
MAX_WORKERS = 8

final_df_list = []  # collect per-season concatenations here
all_failed_games = {} #Keep Track of Failed Games Across all seasons on SEASONS

for szn in SEASONS:
    print(f"\nNow Running Season: {szn}")
    _, game_ids = gamelog_gameid(szn)

    time.sleep(5) #Sleep for 5 seconds after first API call for Game Log for a Season

    season_frames = []
    failures = []

    # Parallelize over games in this season
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(process_game, gid, szn): gid for gid in game_ids}

        # Progress bar advances as futures complete (order-agnostic)
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Season {szn}"):
            gid = futures[fut]
            try:
                season_frames.append(fut.result())
            except Exception as e:
                failures.append((gid, e))

    # Concatenate once per season for speed
    if season_frames:
        season_df = pd.concat(season_frames, axis=0, ignore_index=True)
        final_df_list.append(season_df)

    if failures:
        # You can log these and optionally retry
        print(f"[WARN] {len(failures)} games failed in {szn}. Examples:")
        for gid, err in failures[:5]:
            print(f"  - {gid}: {err}")

        #Append All Failures to all_failed_games
        all_failed_games[szn] = failures

    # If you still want a short pause between seasons, keep this:
    time.sleep(5)

# Single concat at the very end
final_df = pd.concat(final_df_list, axis=0, ignore_index=True)



Now Running Season: 2022-23


100%|██████████████████████████████████████████| 27/27 [00:00<00:00, 760.03it/s]

100%|██████████████████████████████████████████| 36/36 [00:00<00:00, 627.06it/s]

100%|██████████████████████████████████████████| 37/37 [00:00<00:00, 811.30it/s]

Season 2022-23:   2%|▌                        | 30/1230 [00:37<19:50,  1.01it/s]

100%|██████████████████████████████████████████| 31/31 [00:00<00:00, 842.00it/s]

100%|██████████████████████████████████████████| 34/34 [00:00<00:00, 781.19it/s]

100%|██████████████████████████████████████████| 27/27 [00:00<00:00, 861.02it/s]

100%|██████████████████████████████████████████| 34/34 [00:00<00:00, 825.92it/s]

100%|██████████████████████████████████████████| 36/36 [00:00<00:00, 788.68it/s]

100%|██████████████████████████████████████████| 33/33 [00:00<00:00, 567.21it/s]

100%|██████████████████████████████████████████| 33/33 [00:00<00:00, 788.65it/s]

100%|██████████████████████████████████████████| 23/23 [00:00<00:00, 821.08it/s]

100%|███████████

[WARN] 113 games failed in 2022-23. Examples:
  - 0022200039: index 0 is out of bounds for axis 0 with size 0
  - 0022200497: index 0 is out of bounds for axis 0 with size 0
  - 0022201167: index 0 is out of bounds for axis 0 with size 0
  - 0022201072: index 0 is out of bounds for axis 0 with size 0
  - 0022200869: index 0 is out of bounds for axis 0 with size 0

Now Running Season: 2023-24


100%|██████████████████████████████████████████| 27/27 [00:00<00:00, 774.82it/s]

Season 2023-24:   0%|                          | 5/1230 [00:13<38:43,  1.90s/it]

100%|██████████████████████████████████████████| 29/29 [00:00<00:00, 659.35it/s]

100%|██████████████████████████████████████████| 29/29 [00:00<00:00, 784.67it/s]

100%|██████████████████████████████████████████| 30/30 [00:00<00:00, 843.05it/s]

100%|██████████████████████████████████████████| 34/34 [00:00<00:00, 437.60it/s]


100%|██████████████████████████████████████████| 24/24 [00:00<00:00, 726.52it/s]

Season 2023-24:   4%|█                        | 50/1230 [01:30<56:53,  2.89s/it]

100%|██████████████████████████████████████████| 30/30 [00:00<00:00, 530.11it/s]

100%|██████████████████████████████████████████| 29/29 [00:00<00:00, 560.91it/s]

Season 2023-24:   6%|█▎                     | 73/1230 [02:22<1:16:09,  3.95s/it]

100%|██████████████████████████████████████████| 32/32 [00:00<00:00, 728.06it/s]

100%|██████████

[WARN] 208 games failed in 2023-24. Examples:
  - 0022300699: index 0 is out of bounds for axis 0 with size 0
  - 0022300088: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
  - 0022301050: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
  - 0022300498: index 0 is out of bounds for axis 0 with size 0
  - 0022300923: index 0 is out of bounds for axis 0 with size 0

Now Running Season: 2024-25


100%|██████████████████████████████████████████| 32/32 [00:00<00:00, 805.17it/s]

100%|██████████████████████████████████████████| 32/32 [00:00<00:00, 811.21it/s]

100%|██████████████████████████████████████████| 35/35 [00:00<00:00, 834.48it/s]

100%|██████████████████████████████████████████| 26/26 [00:00<00:00, 780.25it/s]

100%|██████████████████████████████████████████| 29/29 [00:00<00:00, 761.89it/s]

100%|██████████████████████████████████████████| 25/25 [00:00<00:00, 769.21it/s]

100%|██████████████████████████████████████████| 31/31 [00:00<00:00, 762.03it/s]

100%|██████████████████████████████████████████| 29/29 [00:00<00:00, 754.76it/s]

100%|██████████████████████████████████████████| 26/26 [00:00<00:00, 579.84it/s]

100%|██████████████████████████████████████████| 24/24 [00:00<00:00, 834.63it/s]

100%|██████████████████████████████████████████| 35/35 [00:00<00:00, 826.57it/s]

100%|██████████████████████████████████████████| 31/31 [00:00<00:00, 558.94it/s]


100%|██████████

[WARN] 1012 games failed in 2024-25. Examples:
  - 0022401011: index 0 is out of bounds for axis 0 with size 0
  - 0022400223: index 0 is out of bounds for axis 0 with size 0
  - 0022400290: index 0 is out of bounds for axis 0 with size 0
  - 0022400390: index 0 is out of bounds for axis 0 with size 0
  - 0022401069: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)


## Save the Failed Games to a .pkl File

In [51]:
import pickle

with open("failed_games.pkl", "wb") as file:
    pickle.dump(all_failed_games, file)

## Save the Pulled Data to a .CSV File

In [53]:
final_df.to_csv("stint_wise_data.csv", index = False)

# Load the Failed Games .pkl File and Rerun the Scraping

In [73]:
failed_games_df = pd.DataFrame(columns = ['SEASON', 'GAME_ID', 'STINT_START_TIME', 'STINT_END_TIME', 'PT_DIFF'])

In [75]:
import pickle

# Replace with the path to the latest failed games .pkl file
# Use 'failed_games_2025-11-30.pkl' for next Run
file_path = 'failed_games.pkl' 

with open(file_path, 'rb') as file:
    all_failed_games = pickle.load(file)

all_failed_games

{'2022-23': [('0022200039',
   IndexError('index 0 is out of bounds for axis 0 with size 0')),
  ('0022200497',
   IndexError('index 0 is out of bounds for axis 0 with size 0')),
  ('0022201167',
   IndexError('index 0 is out of bounds for axis 0 with size 0')),
  ('0022201072',
   IndexError('index 0 is out of bounds for axis 0 with size 0')),
  ('0022200869',
   IndexError('index 0 is out of bounds for axis 0 with size 0')),
  ('0022200281',
   IndexError('index 0 is out of bounds for axis 0 with size 0')),
  ('0022200793',
   json.decoder.JSONDecodeError('Expecting value: line 1 column 1 (char 0)')),
  ('0022200426',
   IndexError('index 0 is out of bounds for axis 0 with size 0')),
  ('0022200969',
   IndexError('index 0 is out of bounds for axis 0 with size 0')),
  ('0022200892',
   IndexError('index 0 is out of bounds for axis 0 with size 0')),
  ('0022200397',
   json.decoder.JSONDecodeError('Expecting value: line 1 column 1 (char 0)')),
  ('0022201194',
   IndexError('index 0 i

In [77]:
all_failed_games.keys()

dict_keys(['2022-23', '2023-24', '2024-25'])

In [79]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings("ignore")

# Tune this based on your API limits / machine
MAX_WORKERS = 8

failed_games_df_list = []  # collect per-season concatenations here
new_failed_games = {} # Record any new failed games here

for szn in all_failed_games.keys():
    print(f"\nNow Running Season: {szn}")

    game_ids = [item[0] for item in all_failed_games[szn]] #Grab the list of Game IDs of the failed Games.
        
    time.sleep(5) #Sleep for 5 seconds after first API call for Game Log for a Season

    season_frames = []
    failures = []

    # Parallelize over games in this season
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(process_game, gid, szn): gid for gid in game_ids}

        # Progress bar advances as futures complete (order-agnostic)
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Season {szn}"):
            gid = futures[fut]
            try:
                season_frames.append(fut.result())
            except Exception as e:
                failures.append((gid, e))

    # Concatenate once per season for speed
    if season_frames:
        season_df = pd.concat(season_frames, axis=0, ignore_index=True)
        failed_games_df_list.append(season_df)

    if failures:
        # Log Failures
        print(f"[WARN] {len(failures)} games failed in {szn}. Examples:")
        for gid, err in failures[:5]:
            print(f"  - {gid}: {err}")

        #Log Failures
        new_failed_games[szn] = failures

    # If you still want a short pause between seasons, keep this:
    time.sleep(5)

# Single concat at the very end
failed_games_df = pd.concat(failed_games_df_list, axis=0, ignore_index=True)



Now Running Season: 2022-23


100%|██████████████████████████████████████████| 36/36 [00:00<00:00, 771.35it/s]

100%|██████████████████████████████████████████| 26/26 [00:00<00:00, 478.73it/s]

Season 2022-23: 100%|█████████████████████████| 113/113 [03:12<00:00,  1.70s/it]


[WARN] 80 games failed in 2022-23. Examples:
  - 0022200869: index 0 is out of bounds for axis 0 with size 0
  - 0022201072: index 0 is out of bounds for axis 0 with size 0
  - 0022200281: index 0 is out of bounds for axis 0 with size 0
  - 0022200426: index 0 is out of bounds for axis 0 with size 0
  - 0022201167: index 0 is out of bounds for axis 0 with size 0

Now Running Season: 2023-24


100%|██████████████████████████████████████████| 30/30 [00:00<00:00, 542.74it/s]

100%|██████████████████████████████████████████| 27/27 [00:00<00:00, 493.65it/s]

Season 2023-24:  35%|█████████                 | 72/208 [01:34<02:46,  1.23s/it]

100%|██████████████████████████████████████████| 29/29 [00:00<00:00, 861.79it/s]

Season 2023-24:  36%|█████████▎                | 74/208 [01:37<03:06,  1.39s/it]

100%|██████████████████████████████████████████| 28/28 [00:00<00:00, 778.28it/s]

100%|██████████████████████████████████████████| 27/27 [00:00<00:00, 828.07it/s]

Season 2023-24:  44%|███████████▍              | 91/208 [01:59<02:36,  1.34s/it]

100%|██████████████████████████████████████████| 28/28 [00:00<00:00, 867.85it/s]

100%|██████████████████████████████████████████| 40/40 [00:00<00:00, 629.71it/s]

100%|██████████████████████████████████████████| 35/35 [00:00<00:00, 770.98it/s]

Season 2023-24:  54%|█████████████▌           | 113/208 [02:27<01:43,  1.08s/it]

100%|███████████

[WARN] 67 games failed in 2023-24. Examples:
  - 0022300506: index 0 is out of bounds for axis 0 with size 0
  - 0022300923: index 0 is out of bounds for axis 0 with size 0
  - 0022301050: index 0 is out of bounds for axis 0 with size 0
  - 0022300699: index 0 is out of bounds for axis 0 with size 0
  - 0022300498: index 0 is out of bounds for axis 0 with size 0

Now Running Season: 2024-25


Season 2024-25:   1%|▏                         | 6/1012 [00:13<26:16,  1.57s/it]

100%|██████████████████████████████████████████| 26/26 [00:00<00:00, 831.63it/s]

100%|██████████████████████████████████████████| 31/31 [00:00<00:00, 816.28it/s]

100%|██████████████████████████████████████████| 41/41 [00:00<00:00, 794.21it/s]

100%|██████████████████████████████████████████| 32/32 [00:00<00:00, 782.93it/s]

100%|██████████████████████████████████████████| 36/36 [00:00<00:00, 841.91it/s]

100%|██████████████████████████████████████████| 26/26 [00:00<00:00, 535.37it/s]

100%|██████████████████████████████████████████| 31/31 [00:00<00:00, 440.51it/s]


Season 2024-25:   6%|█▌                       | 63/1012 [01:36<22:27,  1.42s/it]


100%|██████████████████████████████████████████| 22/22 [00:00<00:00, 696.64it/s]

100%|██████████████████████████████████████████| 29/29 [00:00<00:00, 852.24it/s]

100%|██████████████████████████████████████████| 25/25 [00:00<00:00, 792.02it/s]

100%|█████████

[WARN] 83 games failed in 2024-25. Examples:
  - 0022400223: index 0 is out of bounds for axis 0 with size 0
  - 0022400390: index 0 is out of bounds for axis 0 with size 0
  - 0022400290: index 0 is out of bounds for axis 0 with size 0
  - 0022400209: index 0 is out of bounds for axis 0 with size 0
  - 0022401011: index 0 is out of bounds for axis 0 with size 0


## Save any New Failed Games into a new .pkl File

In [81]:
#Save any failed games to a .pkl file
import pickle
from datetime import datetime

with open(f"failed_games_{datetime.now().date()}.pkl", "wb") as file:
    pickle.dump(new_failed_games, file)

## Save the Results of failed_games_df into a .CSV

In [83]:
from datetime import datetime

failed_games_df.to_csv(f"stint_wise_data_from_failed_games_{datetime.now().date()}.csv", index = False)